# Gemma 4 E4B IT — layer 32 confirmatory native-readout validation

Layer 32 is **not** relabelled as validated after the fact. It gets a new test,
specified in full before it runs.

The v2 early-layer recalibration fitted layers 20, 26, 32 and 38 on Gemma-IT
chat-formatted text and validated the native readout on 8 held-out prompts.
Layer 38 passed. Layer 32 scored a median rank of 1.0 and an MRR of 1.0 — but
failed the run's conjunction criterion, because its mean top-10 overlap (0.113)
did not beat the wrong-layer control (0.150). Eight prompts cannot settle that.

This notebook reuses the **saved, frozen** v2 lens and evaluates layer 32 alone
on **32 genuinely new held-out text prompts**, under a criterion predeclared
before any result-producing cell. Top-10 overlap is reported as a secondary
metric and cannot veto a primary-metric pass.

**It never fits a lens.** There is no call to `jlens.fitting.fit` anywhere in
this notebook. `lens.validated.pt`, `validated_lens_manifest.json`,
`native_readout_validation.json`, `heldout_validation.json` and the original
calibration state are opened read-only and never rewritten.

**It never reads SpokenCOCO, images, audio, or any multimodal activation.**
The prompts come from WikiText-103 rendered with Gemma's official IT chat
template, exactly as the v2 calibration rendered its own.

The verdict is exactly one of `VALIDATED_FOR_MULTIMODAL_FOLLOWUP` or
`LAYER32_CONFIRMATORY_NO_GO`. The multimodal experiment's selected layer is
**not** changed here; that decision comes after this result.

In [ ]:
# 1. Primitive bootstrap constants only — nothing imported from this project yet
REPO_URL = "https://github.com/MechInterpreter/jacobian-lens-gemma.git"
REPO_BRANCH = "experiment/spokencoco-jspace-pilot"
REPO_DIR = "/content/jacobian-lens-gemma"
print(REPO_URL, REPO_BRANCH, REPO_DIR, sep="\n")

In [ ]:
# 2. Idempotent checkout and editable install, then verify the package imports
import os, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    repo = Path(REPO_DIR)
    if not (repo / ".git").exists():
        subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "checkout", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{REPO_BRANCH}"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", REPO_DIR], check=True)
    os.chdir(REPO_DIR)


def _git(*args):
    """HEAD facts, or None outside a checkout. Provenance must not crash the run."""
    try:
        out = subprocess.run(["git", *args], capture_output=True, text=True, timeout=30)
    except (OSError, subprocess.SubprocessError):
        return None
    return out.stdout.strip() if out.returncode == 0 else None


BRANCH = _git("branch", "--show-current")
COMMIT = _git("rev-parse", "HEAD")
if IN_COLAB and BRANCH != REPO_BRANCH:
    raise RuntimeError(f"expected {REPO_BRANCH}, checked out {BRANCH}")

# Project modules are imported only after the editable install above.
import jlens

print(f"branch={BRANCH}\ncommit={COMMIT}\njlens={jlens.__file__}")

In [ ]:
# 3. Configuration — the real model never loads automatically
RUN_CONFIRMATORY_VALIDATION = False  # change manually in Colab, on an L4

# The v2 artifacts this run consumes, read-only.
DRIVE_RUNS_ROOT = "/content/drive/MyDrive/jacobian-lens-gemma/runs"
V2_RUN_NAME = "text_jlens_early_layer_recalibration_v2"
LENS_CHECKSUM = "sha256:4b17bf6086901e633f94d3391f5de6eccd3e735cc24cece63887505d73641c2b"
MODEL_REVISION = "fa62d88df2e6df5efa9d26ad6b3beaea2765f0cd"
CONFIG_PATH = "configs/gemma_text_early_layer_recalibration.yaml"

# Candidate pool size. The v2 run consumed the first 40 WikiText records
# (32 fitting + 8 validation); a wider pool leaves a real choice for the seed
# to make after those 40 are excluded by hash.
POOL_RECORDS = 160

from jlens.metadata import load_config
from jlens.native_readout import (
    CONFIRMATORY_CRITERION,
    CONFIRMATORY_LAYER,
    CONFIRMATORY_PROMPT_SEED,
    CONFIRMATORY_PROTOCOL,
    CONTROL_SEED,
    CONTROL_VARIANTS,
    CRITERION_TEXT,
    DIAGNOSTIC_VARIANTS,
    N_CONFIRMATORY_PROMPTS,
    VERDICT_NO_GO,
    VERDICT_VALIDATED,
)

CONFIG = load_config(CONFIG_PATH)
LAYER = CONFIRMATORY_LAYER  # 32, and only 32

if CONFIG["model"]["revision"] != MODEL_REVISION:
    raise RuntimeError(
        f"config pins revision {CONFIG['model']['revision']}, this run expects {MODEL_REVISION}"
    )
if LAYER not in CONFIG["sites"]["source_layers"]:
    raise RuntimeError(f"layer {LAYER} is not among the fitted layers {CONFIG['sites']['source_layers']}")

print(f"RUN_CONFIRMATORY_VALIDATION={RUN_CONFIRMATORY_VALIDATION}")
print(f"layer under test = {LAYER} (only)")
print(f"held-out prompts = {N_CONFIRMATORY_PROMPTS}, prompt seed = {CONFIRMATORY_PROMPT_SEED}")
print(f"controls = {list(CONTROL_VARIANTS)}, control seed = {CONTROL_SEED}")
print(f"diagnostic (non-blocking) = {list(DIAGNOSTIC_VARIANTS)}")
print(f"protocol = {CONFIRMATORY_PROTOCOL}")
print("no lens is fitted by this notebook; the saved v2 artifact is read-only")

## The predeclared pass criterion

The next cell prints the criterion in full. It runs **before** any cell that
loads the model, selects prompts, or produces a number, and the criterion is a
module constant (`jlens.native_readout.CONFIRMATORY_CRITERION`) rather than a
value typed into a result cell — so it cannot be quietly adjusted once results
exist. Its checksum is bound into the resume fingerprint: editing the criterion
invalidates every stored result rather than rescoring old rows under a new rule.

Top-10 overlap is a **secondary** metric here. It is reported in every artifact
and in the report table, and it appears in no check. That is the deliberate
difference from the v2 conjunction, and the reason this is a new test rather
than a reinterpretation of the old one.

In [ ]:
# 4. Print the predeclared criterion before any result-producing cell
print(CRITERION_TEXT)
print("criterion digest:", CONFIRMATORY_CRITERION.digest)

In [ ]:
# 5. Mount Drive (only after package bootstrap) and locate the v2 artifacts
RUN_DIR = ARTIFACT_DIR = CONFIRMATORY_DIR = None
if RUN_CONFIRMATORY_VALIDATION:
    if not IN_COLAB:
        raise RuntimeError("the confirmatory run is intended for Colab on an NVIDIA L4")
    from google.colab import drive

    drive.mount("/content/drive")
    RUN_DIR = Path(DRIVE_RUNS_ROOT) / V2_RUN_NAME
    ARTIFACT_DIR = RUN_DIR / "artifacts"
    CONFIRMATORY_DIR = ARTIFACT_DIR / "layer32_confirmatory_validation"
    if not ARTIFACT_DIR.is_dir():
        raise RuntimeError(f"{ARTIFACT_DIR} not found; this run consumes the saved v2 artifacts")
    if "spokencoco" in str(RUN_DIR).lower() or "multimodal" in str(RUN_DIR).lower():
        raise RuntimeError(f"{RUN_DIR} is not a text-only run directory")
    print("v2 run directory:", RUN_DIR)
    print("confirmatory output directory:", CONFIRMATORY_DIR)
else:
    print("Drive mount skipped; set RUN_CONFIRMATORY_VALIDATION=True manually")

In [ ]:
# 6. Load the saved lens read-only and verify every precondition
LENS = LENS_RECORD = V2_MANIFEST = V2_PROMPT_METADATA = None
if RUN_CONFIRMATORY_VALIDATION:
    import json

    from jlens.lens import JacobianLens
    from jlens.native_readout import verify_saved_lens

    lens_path = ARTIFACT_DIR / "lens.validated.pt"
    manifest_path = ARTIFACT_DIR / "validated_lens_manifest.json"
    native_path = ARTIFACT_DIR / "native_readout_validation.json"
    heldout_path = ARTIFACT_DIR / "heldout_validation.json"
    prompt_meta_path = RUN_DIR / "prompt_metadata.json"

    for required in (lens_path, manifest_path, native_path, prompt_meta_path):
        if not required.is_file():
            raise RuntimeError(f"missing required v2 artifact: {required}")
    print("original validation manifest present:", manifest_path)
    print("original native readout results present:", native_path)
    print("original reconstruction diagnostics present:", heldout_path.is_file())

    V2_MANIFEST = json.loads(manifest_path.read_text(encoding="utf-8"))
    V2_PROMPT_METADATA = json.loads(prompt_meta_path.read_text(encoding="utf-8"))

    # Read-only load. Nothing below writes to lens_path.
    LENS = JacobianLens.load(str(lens_path))
    LENS_RECORD = verify_saved_lens(
        LENS,
        lens_path=lens_path,
        expected_checksum=LENS_CHECKSUM,
        manifest=V2_MANIFEST,
        expected_model_repo_id=CONFIG["model"]["repo_id"],
        expected_model_revision=MODEL_REVISION,
        expected_d_model=CONFIG["model"]["expect_d_model"],
        expected_hook_site=CONFIG["sites"]["source_site"],
        layer=LAYER,
    )
    for check in LENS_RECORD["checks"]:
        print(f"  OK  {check['check']}: {check['detail']}")
    print(json.dumps({k: v for k, v in LENS_RECORD.items() if k != "checks"}, indent=2))

In [ ]:
# 7. Authenticate and load the exact pinned checkpoint
MODEL = LOAD_INFO = ARCH = None
if RUN_CONFIRMATORY_VALIDATION:
    import torch
    from google.colab import userdata
    from huggingface_hub import login

    token = userdata.get("HF_TOKEN")
    if not token:
        raise RuntimeError("Add an HF_TOKEN Colab secret with Gemma access")
    login(token=token, add_to_git_credential=False)

    from jlens.gemma4 import load_gemma4, verify_architecture

    MODEL, LOAD_INFO = load_gemma4(
        repo_id=CONFIG["model"]["repo_id"],
        revision=MODEL_REVISION,
        dtype=torch.bfloat16,
        device_map="cuda",
        allow_model_load=True,
        token=token,
    )
    ARCH = verify_architecture(
        MODEL,
        expect_n_layers=CONFIG["model"]["expect_n_layers"],
        expect_d_model=CONFIG["model"]["expect_d_model"],
        expect_vocab_size=CONFIG["model"]["expect_vocab_size"],
    )
    if LOAD_INFO["model_revision"] != MODEL_REVISION:
        raise RuntimeError(
            f"loaded revision {LOAD_INFO['model_revision']} != pinned {MODEL_REVISION}"
        )
    print(ARCH)

In [ ]:
# 8. Build 32 genuinely new held-out prompts, refusing any overlap
HELDOUT_PROMPTS = PROMPT_MANIFEST = None
if RUN_CONFIRMATORY_VALIDATION:
    from jlens.examples import load_wikitext_prompts
    from jlens.native_readout import excluded_prompt_hashes, select_confirmatory_prompts

    instruction = CONFIG["recalibration"]["chat_instruction"]
    max_tokens = CONFIG["fitting"]["max_seq_len"]

    def render(passage):
        """The v2 rendering, character for character.

        Shortens only the passage, never right-truncating the assistant
        generation suffix. Any drift here would change the hashes and silently
        defeat the overlap check, so the identity is asserted below rather than
        assumed.
        """
        passage = passage.strip()
        while True:
            text = MODEL.tokenizer.apply_chat_template(
                [{"role": "user", "content": f"{instruction}\n\n{passage}"}],
                tokenize=False,
                add_generation_prompt=True,
            )
            ids = MODEL.tokenizer(text, add_special_tokens=False)["input_ids"]
            if len(ids) + 1 <= max_tokens:
                return text
            if len(passage) <= 64:
                raise RuntimeError("chat template alone exceeds max_seq_len")
            passage = passage[: max(64, int(len(passage) * 0.85))]

    POOL = [render(text) for text in load_wikitext_prompts(POOL_RECORDS)]
    EXCLUDED = excluded_prompt_hashes(V2_PROMPT_METADATA)
    HELDOUT_PROMPTS, PROMPT_MANIFEST = select_confirmatory_prompts(
        POOL,
        n_prompts=N_CONFIRMATORY_PROMPTS,
        excluded=EXCLUDED,
        seed=CONFIRMATORY_PROMPT_SEED,
    )

    # The v2 prompts are the first 40 records of the same stream. If our
    # rendering still matches v2's, exactly those 40 are excluded by hash. A
    # different count means the rendering drifted and the overlap check is no
    # longer trustworthy — refuse rather than report a "held-out" result.
    n_v2 = CONFIG["fitting"]["n_prompts"] + CONFIG["recalibration"]["heldout_prompts"]
    excluded_indices = {row["pool_index"] for row in PROMPT_MANIFEST["excluded"]}
    if excluded_indices != set(range(n_v2)):
        raise RuntimeError(
            f"expected the first {n_v2} pool records to be the v2 prompts, but the "
            f"hash-excluded records were {sorted(excluded_indices)}; the chat "
            "rendering no longer reproduces the v2 prompts, so overlap cannot be ruled out"
        )

    PROMPT_MANIFEST["source"] = "Salesforce/wikitext wikitext-103-raw-v1 (train, streaming order)"
    PROMPT_MANIFEST["chat_protocol"] = CONFIG["recalibration"]["protocol"]
    PROMPT_MANIFEST["chat_instruction"] = instruction
    PROMPT_MANIFEST["max_seq_len"] = max_tokens
    PROMPT_MANIFEST["modality"] = "text"
    PROMPT_MANIFEST["excluded_roles"] = sorted(set(EXCLUDED.values()))
    print(
        f"selected {len(HELDOUT_PROMPTS)} new held-out prompts from a pool of "
        f"{PROMPT_MANIFEST['pool_size']}; {PROMPT_MANIFEST['n_excluded']} excluded as "
        f"v2 fitting/validation prompts; overlap=0"
    )

In [ ]:
# 9. Open the resumable output directory under a fingerprint of every input
STORE = RESUME_STATUS = None
if RUN_CONFIRMATORY_VALIDATION:
    from jlens.metadata import environment_manifest
    from jlens.native_readout import ConfirmatoryFingerprint, ConfirmatoryStore

    FINGERPRINT = ConfirmatoryFingerprint(
        protocol=CONFIRMATORY_PROTOCOL,
        lens_checksum=LENS_RECORD["lens_checksum"],
        model_repo_id=CONFIG["model"]["repo_id"],
        model_revision=LOAD_INFO["model_revision"],
        prompt_protocol=PROMPT_MANIFEST["chat_protocol"],
        prompt_seed=CONFIRMATORY_PROMPT_SEED,
        prompt_hashes=tuple(row["prompt_sha256"] for row in PROMPT_MANIFEST["prompts"]),
        layer=LAYER,
        controls=CONTROL_VARIANTS,
        control_seed=CONTROL_SEED,
        criterion_digest=CONFIRMATORY_CRITERION.digest,
    )
    STORE = ConfirmatoryStore(CONFIRMATORY_DIR, FINGERPRINT)
    RESUME_STATUS = STORE.open()  # raises IncompatibleStateError on any change
    print(f"run state: {RESUME_STATUS}")

    STORE.write_artifact(
        "config.json",
        {
            "protocol": CONFIRMATORY_PROTOCOL,
            "layer": LAYER,
            "layers_evaluated": [LAYER],
            "n_prompts": N_CONFIRMATORY_PROMPTS,
            "prompt_seed": CONFIRMATORY_PROMPT_SEED,
            "control_seed": CONTROL_SEED,
            "controls": list(CONTROL_VARIANTS),
            "diagnostics": list(DIAGNOSTIC_VARIANTS),
            "primary_metrics": list(CONFIRMATORY_CRITERION.primary_metrics),
            "secondary_metrics": list(CONFIRMATORY_CRITERION.secondary_metrics),
            "criterion": CONFIRMATORY_CRITERION.to_dict(),
            "criterion_digest": CONFIRMATORY_CRITERION.digest,
            "criterion_text": CRITERION_TEXT,
            "fits_a_lens": False,
            "source_lens": LENS_RECORD,
            "v2_manifest": V2_MANIFEST,
            "repo_branch": BRANCH,
            "repo_commit": COMMIT,
            "environment": environment_manifest(),
            "fingerprint": FINGERPRINT.to_dict(),
        },
    )
    STORE.write_artifact("prompt_manifest.json", PROMPT_MANIFEST)
    print("wrote config.json and prompt_manifest.json")

In [ ]:
# 10. Score layer 32 on each held-out prompt, one atomic result per prompt
ROWS = None
if RUN_CONFIRMATORY_VALIDATION:
    import torch

    from jlens.controls import wrong_layer_mapping
    from jlens.hooks import ActivationRecorder
    from jlens.native_readout import build_readout_variants, native_readout_row

    VARIANTS = build_readout_variants(LENS, seed=CONTROL_SEED)
    WRONG_LAYER_MAP = wrong_layer_mapping(LENS.source_layers)
    print(f"wrong-layer control at L{LAYER} uses the J fitted at L{WRONG_LAYER_MAP[LAYER]}")

    final_layer = MODEL.n_layers - 1
    ROWS, reused, computed = [], 0, 0
    for index, prompt in enumerate(HELDOUT_PROMPTS):
        sha = PROMPT_MANIFEST["prompts"][index]["prompt_sha256"]
        cached = STORE.load_result(index, sha)
        if cached is not None:
            ROWS.extend(cached["rows"])
            reused += 1
            continue

        ids = MODEL.encode(prompt, max_length=CONFIG["fitting"]["max_seq_len"])
        with torch.no_grad():
            with ActivationRecorder(MODEL.layers, at=[LAYER, final_layer]) as recorder:
                MODEL.forward(ids)
            actual = MODEL.unembed(recorder.activations[final_layer][0, -1].float())
            residual = recorder.activations[LAYER][0, -1].float()
            scored = {"logit_lens": MODEL.unembed(residual)}
            for name, variant in VARIANTS.items():
                scored[name] = MODEL.unembed(variant.transport(residual, LAYER))

        rows = [
            native_readout_row(
                sample_index=index,
                prompt_sha=sha,
                layer=LAYER,
                variant=name,
                variant_logits=scored[name],
                actual_logits=actual,
            )
            for name in ("j_lens", *CONTROL_VARIANTS, *DIAGNOSTIC_VARIANTS)
        ]
        STORE.save_result(index, sha, {"sample": index, "prompt_sha256": sha, "rows": rows})
        ROWS.extend(rows)
        computed += 1
        del recorder, actual, residual, scored
        torch.cuda.empty_cache()
        print(f"prompt {index + 1}/{len(HELDOUT_PROMPTS)} computed")

    print(f"prompt results: reused={reused} computed={computed} total={reused + computed}")

In [ ]:
# 11. Aggregate, apply the predeclared criterion, and save the verdict atomically
VERDICT = None
if RUN_CONFIRMATORY_VALIDATION:
    from jlens.native_readout import confirmatory_report_markdown, evaluate_confirmatory

    VERDICT = evaluate_confirmatory(ROWS, criterion=CONFIRMATORY_CRITERION, layer=LAYER)
    METRICS = VERDICT["metrics"]

    STORE.write_artifact(
        "aggregate_metrics.json",
        {
            "protocol": CONFIRMATORY_PROTOCOL,
            "layer": LAYER,
            "n_prompts": METRICS["j_lens"]["n_prompts"],
            "j_lens": METRICS["j_lens"],
            "logit_lens_diagnostic": METRICS["logit_lens"],
            "rows": ROWS,
        },
    )
    STORE.write_artifact(
        "control_metrics.json",
        {
            "protocol": CONFIRMATORY_PROTOCOL,
            "layer": LAYER,
            "control_seed": CONTROL_SEED,
            "wrong_layer_mapping": {str(k): v for k, v in WRONG_LAYER_MAP.items()},
            "controls": {name: METRICS[name] for name in CONTROL_VARIANTS},
            "diagnostics_not_blocking": {name: METRICS[name] for name in DIAGNOSTIC_VARIANTS},
        },
    )
    STORE.write_artifact("verdict.json", VERDICT)
    STORE.write_artifact(
        "report.md",
        confirmatory_report_markdown(
            VERDICT, prompt_manifest=PROMPT_MANIFEST, lens_record=LENS_RECORD
        ),
    )

    for name, row in METRICS.items():
        print(
            f"  {name:12s} top1={row['top1_agreement']:.4f} MRR={row['mean_reciprocal_rank']:.5f} "
            f"median-rank={row['median_target_rank']:8.1f} top10*={row['mean_top10_overlap']:.3f}"
        )
    print("  * secondary, non-blocking")
    for check in VERDICT["checks"]:
        print(f"  {'PASS' if check['passed'] else 'FAIL'}  {check['check']}: {check['detail']}")
    print()
    print("VERDICT:", VERDICT["verdict"])

In [ ]:
# 12. Publish a separate confirmatory manifest, or preserve the NO-GO
if RUN_CONFIRMATORY_VALIDATION:
    import json

    # The original manifests are never rewritten, on either branch.
    if VERDICT["verdict"] == VERDICT_VALIDATED:
        CONFIRMATORY_MANIFEST = {
            "status": "layer32_independently_confirmed",
            "protocol": CONFIRMATORY_PROTOCOL,
            "confirms_layer": LAYER,
            "lens_path": LENS_RECORD["lens_path"],
            "lens_checksum": LENS_RECORD["lens_checksum"],
            "model_repo_id": CONFIG["model"]["repo_id"],
            "model_revision": LOAD_INFO["model_revision"],
            "original_manifest_path": str(ARTIFACT_DIR / "validated_lens_manifest.json"),
            "original_manifest_unmodified": True,
            "original_native_validation_path": str(ARTIFACT_DIR / "native_readout_validation.json"),
            "n_heldout_prompts": VERDICT["metrics"]["j_lens"]["n_prompts"],
            "prompt_seed": CONFIRMATORY_PROMPT_SEED,
            "prompt_manifest_path": str(CONFIRMATORY_DIR / "prompt_manifest.json"),
            "verdict_path": str(CONFIRMATORY_DIR / "verdict.json"),
            "verdict": VERDICT["verdict"],
            "criterion": CONFIRMATORY_CRITERION.to_dict(),
            "criterion_digest": CONFIRMATORY_CRITERION.digest,
            "metrics": VERDICT["metrics"],
            "note": (
                "Layer 32 is confirmed by this independent test only. The v2 "
                "validated-lens manifest still records the original result and is "
                "left exactly as written."
            ),
        }
        STORE.write_artifact("layer32_confirmatory_manifest.json", CONFIRMATORY_MANIFEST)
        print("published layer32_confirmatory_manifest.json")
        print(json.dumps({k: CONFIRMATORY_MANIFEST[k] for k in ("status", "confirms_layer", "verdict")}, indent=2))
        print()
        print("Layer 32 may join layer 38 in the four-concept follow-up.")
    else:
        print(f"{VERDICT['verdict']}: failed checks {VERDICT['failed_checks']}")
        print("Results are preserved in", CONFIRMATORY_DIR)
        print("Recommendation: keep only layer 38 for the four-concept follow-up.")

    print()
    print("The multimodal experiment's selected layer is unchanged by this notebook.")
    print(json.dumps(STORE.status_report(), indent=2))

## What to send back

Paste the full stdout of the cells whose headers read `# 2`, `# 4`, `# 6`, and
`# 9` through `# 12`. That is the branch and commit, the predeclared criterion
and its digest, the lens verification record, the resume status, the
reused/computed counts, the per-variant metric table, the per-check PASS/FAIL
lines, and the final verdict line.

The verdict line reads exactly one of:

```
VERDICT: VALIDATED_FOR_MULTIMODAL_FOLLOWUP
VERDICT: LAYER32_CONFIRMATORY_NO_GO
```

If the run is interrupted, rerun the notebook from cell 1 with
`RUN_CONFIRMATORY_VALIDATION = True`. It reprints `run state: resuming`, reuses
every checksum-valid prompt result, and computes only what is missing. If it
raises `IncompatibleStateError`, something the results depend on changed — the
message names the field. Do not delete the directory to make the error go away;
send the message back instead.